# Canonical decomposition processes for order-2, -4, and -6 tensors

CDP receives tensor coefficient fields. The notebook never feeds raw dMRI to CDP.

## 1. Prepare the Colab environment

In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

PROJECT_ROOT = Path('/content/dMRI_Interpolation_Repository')
DATA_ROOT = Path('/content/dMRI_Data')
if not PROJECT_ROOT.is_dir():
    from google.colab import files
    print('Upload dMRI_Interpolation_Repository.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise FileNotFoundError('No ZIP archive was uploaded.')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/content')
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Project directory not found: {PROJECT_ROOT}')
if not (DATA_ROOT / 'TENSOR_2x2x2.nii.gz').is_file():
    from google.colab import files
    print('Upload dMRI_Data.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise FileNotFoundError('No dMRI data ZIP archive was uploaded.')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/content')
if not (DATA_ROOT / 'TENSOR_2x2x2.nii.gz').is_file():
    raise FileNotFoundError(f'dMRI data folder not found: {DATA_ROOT}')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(PROJECT_ROOT / 'requirements_colab.txt')
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project ready:', PROJECT_ROOT)
print('External dMRI data ready:', DATA_ROOT)

## 2. Run CDP for each supported tensor order

In [ ]:
import matplotlib.pyplot as plt
from cdp_tensor import FAST_RANKS, cdp_interpolate
from tensor_utils import frobenius_error, load_tensor_field, unsampled_mask

results = {}
for order in (2, 4, 6):
    path = PROJECT_ROOT / 'data' / 'tensors' / f'training_data{order}.mat'
    reference = load_tensor_field(path)
    low = reference[:, ::2, ::2]
    cdp = cdp_interpolate(
        low, reference.shape[1:], order=order, factor=2,
        rank=FAST_RANKS[order], cp_iterations=80,
    )
    metrics = frobenius_error(
        cdp['coefficients'], reference, order=order,
        mask=unsampled_mask(reference.shape[1:], 2),
    )
    results[order] = (reference, cdp, metrics)
    print(
        f"Order {order}, rank {cdp['rank']}: "
        f"{metrics['mean']:.6g} +/- {metrics['std']:.6g}; "
        f"basis error={cdp['cp_relative_error']:.6g}"
    )

## 3. Error maps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for axis, order in zip(axes, (2, 4, 6)):
    image = axis.imshow(results[order][2]['map'], cmap='magma')
    axis.set_title(f'CDP order {order}')
    axis.axis('off')
    fig.colorbar(image, ax=axis, fraction=0.046)
plt.tight_layout()